# Applying GigaTIME to TCGA-BRCA HER2/ERBB2

**Lab meeting notebook**  
Pilot analysis using TCGA-BRCA H&E slides, the released GigaTIME model, and ERBB2 RNA expression.

**Core message:** this is an executable proof-of-work showing that GigaTIME can run on TCGA-BRCA slides and produce virtual mIF/TIME features that can be joined to HER2/ERBB2 expression.

## 1. Setup

Run this notebook from the project root:

```bash
conda activate gigatime-tcga
jupyter lab
```

If using Colab, upload the project outputs or a zip containing `results/gigatime_tcga_brca_extremes/` and `data/tcga_brca/her2_extreme_cases.csv`.

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "results").exists() and (PROJECT_ROOT.parent / "results").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_DIR = PROJECT_ROOT / "results" / "gigatime_tcga_brca_extremes"
SUMMARY_DIR = RESULTS_DIR / "advisor_summary"
HEATMAP_DIR = RESULTS_DIR / "heatmaps"
VIRTUAL_MIF_DIR = PROJECT_ROOT / "docs" / "assets" / "virtual_mif_channels"
VIRTUAL_MIF_COMPOSITE_DIR = PROJECT_ROOT / "docs" / "assets" / "virtual_mif_composites"
DATA_DIR = PROJECT_ROOT / "data" / "tcga_brca"

required = [
    RESULTS_DIR / "slide_scores.csv",
    RESULTS_DIR / "tile_scores.csv",
    SUMMARY_DIR / "advisor_summary.md",
    SUMMARY_DIR / "her2_group_channel_summary.csv",
    SUMMARY_DIR / "joined_slide_her2_gigatime.csv",
    VIRTUAL_MIF_DIR / "virtual_mif_all_channel_group_means.png",
    VIRTUAL_MIF_DIR / "virtual_mif_slide_channel_matrix.png",
    VIRTUAL_MIF_DIR / "her2_high_reference_all_virtual_mif_channels.png",
    VIRTUAL_MIF_DIR / "her2_low_reference_all_virtual_mif_channels.png",
    VIRTUAL_MIF_COMPOSITE_DIR / "her2_high_immune_checkpoint_virtual_mif_montage.png",
    VIRTUAL_MIF_COMPOSITE_DIR / "her2_high_immune_checkpoint_he_vs_virtual_mif.png",
    VIRTUAL_MIF_COMPOSITE_DIR / "her2_low_immune_checkpoint_virtual_mif_montage.png",
    VIRTUAL_MIF_COMPOSITE_DIR / "her2_low_immune_checkpoint_he_vs_virtual_mif.png",
    DATA_DIR / "her2_extreme_cases.csv",
]

missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing expected files:\n" + "\n".join(str(p) for p in missing))

display(Markdown("✅ Required result files found."))

## 2. Research Question

**Question:** Can GigaTIME generate virtual tumor immune microenvironment features from TCGA-BRCA H&E slides, and do these features differ between ERBB2-high and ERBB2-low tumors?

**Important framing:**

- GigaTIME predicts virtual mIF marker maps from H&E.
- HER2 is proxied here by **ERBB2 RNA expression** from TCGA-BRCA.
- This is a replication/adaptation pilot, not a new model and not a clinical HER2 classifier.

## 3. Workflow

```text
TCGA-BRCA H&E diagnostic slides
        ↓
Random tissue tile sampling
        ↓
Released GigaTIME model
        ↓
Virtual mIF / TIME marker activations
        ↓
Slide-level aggregation
        ↓
Join to TCGA ERBB2 expression
        ↓
Compare ERBB2-high vs ERBB2-low groups
```

In [ ]:
joined = pd.read_csv(SUMMARY_DIR / "joined_slide_her2_gigatime.csv")
channel_summary = pd.read_csv(SUMMARY_DIR / "her2_group_channel_summary.csv")
slide_scores = pd.read_csv(RESULTS_DIR / "slide_scores.csv")
tile_scores = pd.read_csv(RESULTS_DIR / "tile_scores.csv")
selected_cases = pd.read_csv(DATA_DIR / "her2_extreme_cases.csv")

cohort_summary = joined.groupby("her2_group").agg(
    cases=("case_submitter_id", "nunique"),
    slides=("slide_id", "count"),
    erbb2_min=("erbb2_tpm", "min"),
    erbb2_median=("erbb2_tpm", "median"),
    erbb2_max=("erbb2_tpm", "max"),
    total_tiles=("n_tiles", "sum"),
    mean_tissue_fraction=("mean_tissue_fraction", "mean"),
).round(4)

display(Markdown("## Pilot Cohort Summary"))
display(cohort_summary)

display(Markdown("## Processed Cases"))
display(joined[["case_submitter_id", "her2_group", "erbb2_tpm", "n_tiles", "mean_tissue_fraction"]]
        .sort_values(["her2_group", "erbb2_tpm"], ascending=[True, False])
        .reset_index(drop=True))

### Speaker Note

The selected cohort was designed from ERBB2 extremes: top and bottom ERBB2 TPM cases from the current TCGA-BRCA pilot metadata. The current run includes the subset of selected slides that successfully downloaded from GDC in time.

## 4. ERBB2 Group Separation

In [ ]:
img = SUMMARY_DIR / "erbb2_tpm_distribution.png"
display(Image(filename=str(img), width=760))

### Speaker Note

This plot shows the ERBB2 expression distribution for the processed subset. The groups are not a clinical HER2 label; they are ERBB2 RNA-expression extremes.

## 5. Virtual mIF Differences by HER2/ERBB2 Group

In [ ]:
display(Image(filename=str(SUMMARY_DIR / "her2_group_channel_boxplots.png"), width=920))

In [ ]:
display(Image(filename=str(SUMMARY_DIR / "her2_group_channel_deltas.png"), width=820))

In [ ]:
cols = [
    "channel",
    "delta_high_minus_low",
    "cohens_d",
    "welch_p_value",
    "mannwhitney_p_value",
    "spearman_erbb2_activation",
    "n_high_slides",
    "n_low_slides",
]
display(channel_summary
        .sort_values("absolute_delta_mean_activation", ascending=False)[cols]
        .round(4)
        .reset_index(drop=True))

### Speaker Note

The current subset is still small, so this should be presented as directionally useful rather than statistically definitive. The goal today is to show that the workflow runs end-to-end and produces interpretable outputs for discussion.

## 6. ERBB2 Correlation Views

In [ ]:
display(Image(filename=str(SUMMARY_DIR / "erbb2_vs_virtual_mif_scatter.png"), width=920))

## 7. All Virtual mIF Channel Images

These figures use the GigaTIME tile-level predictions to show all 23 virtual mIF channels, not only the selected summary markers. The channel maps are model predictions from H&E image tiles; they are not real multiplex immunofluorescence measurements.

- The group-means figure compares average virtual channel activation between ERBB2-high and ERBB2-low processed slides.
- The slide-by-channel matrix shows relative activation patterns across all processed slides.
- The HER2-high and HER2-low reference grids show the spatial tile-level predictions for every GigaTIME channel on one representative slide from each group.


In [ ]:
virtual_mif_figures = [
    ("All 23 virtual mIF channels by ERBB2 group", VIRTUAL_MIF_DIR / "virtual_mif_all_channel_group_means.png", 860),
    ("Slide-by-channel relative activation matrix", VIRTUAL_MIF_DIR / "virtual_mif_slide_channel_matrix.png", 980),
    ("HER2-high reference slide: all virtual mIF channels", VIRTUAL_MIF_DIR / "her2_high_reference_all_virtual_mif_channels.png", 980),
    ("HER2-low reference slide: all virtual mIF channels", VIRTUAL_MIF_DIR / "her2_low_reference_all_virtual_mif_channels.png", 980),
]

for title, figure_path, width in virtual_mif_figures:
    display(Markdown(f"### {title}"))
    display(Image(filename=str(figure_path), width=width))


## 8. Fluorescence-Style Virtual mIF Composites

These images are closer to what a real multiplex immunofluorescence panel looks like. They were generated by rerunning GigaTIME on selected H&E tiles, keeping the full predicted channel maps, and compositing marker colors on a black background.

These are still **virtual predictions**, not experimental mIF measurements.


In [ ]:
virtual_mif_composites = [
    ("HER2-high immune-checkpoint virtual mIF montage", VIRTUAL_MIF_COMPOSITE_DIR / "her2_high_immune_checkpoint_virtual_mif_montage.png", 920),
    ("HER2-high H&E versus virtual mIF", VIRTUAL_MIF_COMPOSITE_DIR / "her2_high_immune_checkpoint_he_vs_virtual_mif.png", 720),
    ("HER2-low immune-checkpoint virtual mIF montage", VIRTUAL_MIF_COMPOSITE_DIR / "her2_low_immune_checkpoint_virtual_mif_montage.png", 920),
    ("HER2-low H&E versus virtual mIF", VIRTUAL_MIF_COMPOSITE_DIR / "her2_low_immune_checkpoint_he_vs_virtual_mif.png", 720),
]

for title, figure_path, width in virtual_mif_composites:
    display(Markdown(f"### {title}"))
    display(Image(filename=str(figure_path), width=width))


## 9. Selected GigaTIME Heatmaps

The heatmaps show tile-level virtual marker activation for selected markers. These are useful visually because they demonstrate that the model produces spatial outputs, not just summary tables.

In [ ]:
heatmaps = sorted(HEATMAP_DIR.glob("*.png"))
display(Markdown(f"Found **{len(heatmaps)}** heatmap PNGs."))

preferred_markers = ["CD3", "CD8", "PD-L1", "CK"]
examples = []
for marker in preferred_markers:
    marker_maps = [p for p in heatmaps if p.stem.endswith(f"_{marker}")]
    examples.extend(marker_maps[:2])

for p in examples[:8]:
    display(Markdown(f"### {p.name}"))
    display(Image(filename=str(p), width=620))

## 10. H&E Slide Images from the Analyzed SVS Files

These panels are generated directly from the TCGA H&E slides used in the GigaTIME run. They are useful for showing what tissue was analyzed, where the sampled tiles came from, and how virtual marker activation relates back to the H&E slide.

In [ ]:
HE_EXAMPLE_DIR = RESULTS_DIR / "he_examples"
he_manifest = pd.read_csv(HE_EXAMPLE_DIR / "he_image_manifest.csv")
display(Markdown(f"Found **{len(he_manifest)}** generated H&E-derived images."))

reference_cases = {
    "HER2-high reference": joined.sort_values("erbb2_tpm", ascending=False).iloc[0]["case_submitter_id"],
    "HER2-low reference": joined.sort_values("erbb2_tpm", ascending=True).iloc[0]["case_submitter_id"],
}

reference_panels = [
    ("he_thumbnail", "Whole-slide H&E thumbnail"),
    ("sampled_tile_overlay", "Analyzed GigaTIME tile locations"),
    ("virtual_CD8_overlay", "H&E + virtual CD8 overlay"),
    ("top_CD8_tile_montage", "Top H&E tiles by virtual CD8"),
    ("virtual_CK_overlay", "H&E + virtual CK overlay"),
    ("top_CK_tile_montage", "Top H&E tiles by virtual CK"),
]

for label, case_id in reference_cases.items():
    case_info = joined[joined["case_submitter_id"] == case_id].iloc[0]
    display(Markdown(
        f"### {label}: `{case_id}` | ERBB2 TPM {case_info['erbb2_tpm']:.1f} | {case_info['her2_group']}"
    ))
    example_rows = he_manifest[he_manifest["case_submitter_id"] == case_id]
    for image_type, title in reference_panels:
        matches = example_rows[example_rows["image_type"] == image_type]
        if matches.empty:
            continue
        row = matches.iloc[0]
        display(Markdown(f"#### {title}"))
        display(Image(filename=str(PROJECT_ROOT / row["path"]), width=900))

### Speaker Note

Use these as reference images in the slides: one HER2-high case and one HER2-low case, each with the whole H&E thumbnail, sampled tile locations, virtual marker overlays, and top-scoring H&E tile montages. The H&E thumbnail gives the whole-slide context; the yellow-box image shows sampled GigaTIME tiles; the colored overlay maps virtual marker activation back onto sampled H&E regions; the tile montage shows high-scoring H&E patches for a selected virtual marker.

## 11. Interpretation for Lab Meeting

**What we can say:**

- The released GigaTIME model runs on TCGA-BRCA diagnostic H&E slides.
- The workflow generates virtual mIF/TIME marker features per slide.
- These features can be joined to TCGA ERBB2 RNA expression.
- The current pilot now supports ERBB2-extreme group summaries and case-level statistics.

**What we should not overclaim:**

- This does not validate HER2 status prediction.
- ERBB2 RNA is not equivalent to clinical HER2 IHC/FISH.
- The current processed subset is still small.
- Tissue QC and full cohort processing are still needed before biological interpretation.

## 12. Next Steps

1. Finish downloading the remaining selected slides from the 20 high / 20 low ERBB2-extreme cohort.
2. Rerun GigaTIME with the same random seed and tile count per slide.
3. Add stronger tile-level tissue/tumor QC.
4. Compare ERBB2 RNA groups with any available clinical HER2 annotations.
5. Decide whether this becomes a replication/adaptation figure for the proposal.

## 13. Optional: Export This Notebook

From the terminal:

```bash
conda activate gigatime-tcga
jupyter nbconvert --to html notebooks/tcga_brca_gigatime_her2_lab_meeting.ipynb
```

You can also take screenshots of the key notebook sections and paste them into slides.